# Snake-RepairLLaMA: RunPod Training Notebook

Use this notebook for:
- **Verification** after running `setup.sh`
- **Quick inference tests** after training
- **Monitoring** training progress

For actual training, use the shell scripts in `tmux`:
```bash
tmux new -s training
bash /workspace/training_codellama/train.sh
```

## 1. System Check

In [ ]:
import os
import torch
import shutil

print("=" * 60)
print("  SYSTEM CHECK")
print("=" * 60)

# GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU: {gpu} ({vram:.1f} GB)")
    if vram >= 40:
        print(f"  Config: batch=8, grad_accum=2 (A6000/A100 mode)")
    elif vram >= 20:
        print(f"  Config: batch=4, grad_accum=4 (RTX 4090 mode)")
    else:
        print(f"  Config: batch=1, grad_accum=16 (low VRAM mode)")
else:
    print("  WARNING: No GPU detected!")

# Disk
stat = shutil.disk_usage("/workspace")
print(f"  Disk: {stat.free / 1e9:.1f} GB free / {stat.total / 1e9:.1f} GB total")

# HF Cache
hf_home = os.environ.get('HF_HOME', '~/.cache/huggingface')
print(f"  HF_HOME: {hf_home}")

# Check persistent volume
print(f"  /workspace exists: {os.path.isdir('/workspace')}")
print("=" * 60)

## 2. Verify Setup (run after setup.sh)

In [ ]:
import os

# Set HF cache (in case notebook kernel doesn't inherit bashrc)
os.environ['HF_HOME'] = '/workspace/huggingface_cache'

checks = {
    "Dataset (train)": os.path.exists("/workspace/dataset/train.parquet"),
    "Dataset (val)": os.path.exists("/workspace/dataset/validation.parquet"),
    "Training script": os.path.exists("/workspace/training_codellama/train_adapter.py"),
    "Inference script": os.path.exists("/workspace/training_codellama/inference.py"),
    "HF cache exists": os.path.isdir("/workspace/huggingface_cache"),
}

print("Setup Verification:")
all_ok = True
for name, ok in checks.items():
    status = "OK" if ok else "MISSING"
    print(f"  [{status}] {name}")
    if not ok:
        all_ok = False

# Check model cache
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('codellama/CodeLlama-7b-Python-hf', local_files_only=True)
    print(f"  [OK] Base model cached (vocab: {len(tok)})")
except:
    print(f"  [MISSING] Base model not cached - run setup.sh")
    all_ok = False

if all_ok:
    print("\nAll checks passed! Ready to train.")
else:
    print("\nSome checks failed. Run: bash /workspace/training_codellama/setup.sh")

## 3. Validate Dataset

In [ ]:
from datasets import load_dataset

TRAIN_PATH = "/workspace/dataset/train.parquet"
VAL_PATH = "/workspace/dataset/validation.parquet"

train_ds = load_dataset("parquet", data_files=TRAIN_PATH, split="train")
val_ds = load_dataset("parquet", data_files=VAL_PATH, split="train")

print(f"Training samples:   {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")
print(f"Columns: {train_ds.column_names}")

# Show samples
print("\nSample examples:")
for i in range(3):
    s = train_ds[i]
    print(f"\n--- Sample {i} ---")
    print(f"Input:  {s['input'][:150]}...")
    print(f"Output: {s['output'][:150]}...")

## 4. Quick 8-bit Load Test
Run this to verify the model loads correctly in INT8 before starting the full training.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

os.environ['HF_HOME'] = '/workspace/huggingface_cache'

print("Testing 8-bit model load...")
quant_config = BitsAndBytesConfig(load_in_8bit=True)

test_model = AutoModelForCausalLM.from_pretrained(
    "codellama/CodeLlama-7b-Python-hf",
    device_map="auto",
    quantization_config=quant_config,
)

print(f"Model loaded! Device: {test_model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# Clean up
del test_model
gc.collect()
torch.cuda.empty_cache()
print("Cleaned up. Ready to train.")

## 5. Monitor Training
Run these while training is happening in tmux.

In [ ]:
# GPU usage
!nvidia-smi

In [ ]:
# Last 20 lines of training log
!tail -20 /workspace/training.log 2>/dev/null || echo "No training log yet."

In [ ]:
# Check saved checkpoints
!ls -lh /workspace/output/codellama-7b-python-adapter/checkpoint-* 2>/dev/null || echo "No checkpoints yet."

In [ ]:
# TensorBoard (access via RunPod proxy: https://<pod-id>-6006.proxy.runpod.net)
%load_ext tensorboard
%tensorboard --logdir /workspace/output/codellama-7b-python-adapter --host 0.0.0.0 --port 6006

## 6. Inference (After Training)
Test your trained adapter on buggy code.

In [ ]:
import os, sys
os.environ['HF_HOME'] = '/workspace/huggingface_cache'
sys.path.insert(0, '/workspace/training_codellama')

from inference import AdapterInference

# Load model + adapter
model = AdapterInference(
    adapter_path="/workspace/output/codellama-7b-python-adapter",
    base_model_name="codellama/CodeLlama-7b-Python-hf",
    quantize="8bit",  # Use 8bit on RunPod, 4bit on local 6GB GPU
)

In [ ]:
# Test with a buggy code sample
buggy_code = """def add(a, b):
# Buggy code:
#     return a - b
<FILL_ME>
    print(result)
"""

result = model.generate_bugfix(buggy_code, num_variations=3)

print("INPUT:")
print(result["input"])
print("\nGENERATED FIXES:")
for i, gen in enumerate(result["generations"], 1):
    print(f"\n--- Variation {i} ---")
    print(gen[:500])

In [ ]:
# Test on actual validation samples
from datasets import load_dataset

val_ds = load_dataset("parquet", data_files="/workspace/dataset/validation.parquet", split="train")

for i in range(5):
    sample = val_ds[i]
    print(f"\n{'='*60}")
    print(f"Sample {i}")
    print(f"{'='*60}")
    print(f"INPUT: {sample['input'][:200]}...")
    print(f"\nEXPECTED: {sample['output'][:200]}")
    
    result = model.generate(sample['input'], max_length=512, temperature=0.3)
    generated = result[0][len(sample['input']):] if result else 'ERROR'
    print(f"\nGENERATED: {generated[:200]}")

## 7. Upload Adapter to HuggingFace Hub

In [ ]:
# Upload trained adapter to HF Hub
!bash /workspace/training_codellama/save_adapter.sh

## 8. Cleanup
Run this before stopping the pod if you want to free disk space.

In [ ]:
# Check disk usage
!echo "Disk usage:"
!du -sh /workspace/huggingface_cache 2>/dev/null || echo "  No HF cache"
!du -sh /workspace/output 2>/dev/null || echo "  No output dir"
!du -sh /workspace/dataset 2>/dev/null || echo "  No dataset"
!df -h /workspace | tail -1 | awk '{print "  Free: " $4}'